# HydraY NNUE — strato nascosto a 1024 sui dati COMPLETI (2,72B)

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~2h30. **Non lasciare la scheda inattiva.**

### A cosa serve
Raddoppia lo strato nascosto, da 512 a 1024, tenendo tutto il resto identico:
stessi 4 king bucket specchiati, stessi 8 bucket di uscita, stesso schedule da
40 superbatch, stesso dataset da 2,72B. L'unica variabile e' la **capacita'**.

### Perche' 1024 E i dati completi insieme
A5 (2026-08-02) ha misurato che passare da 1,18B a 2,72B posizioni non cambia
niente su una rete da 512: **−7,30 ±8,61**. La lettura piu' naturale e' che una
512 su 1,18B fosse gia' satura di capacita' — aggiungere esempi a un modello che
non ha piu' dove metterli non produce nulla.

Una 1024 ha il doppio dei parametri in l0, quindi ha posto. E' esattamente la
condizione in cui i dati in piu' tornano a valere, e per questo va addestrata
sui 2,72B interi e non sul prefisso.

Due risposte al prezzo di una: se vince, hai la rete nuova; se vince di molto,
hai anche la prova che il tetto era la capacita' e non i dati — il che riapre
la questione che A5 sembrava aver chiuso.

### Cosa aspettarsi al ritorno
Il binario paghera' **15-25% di NPS**. Lo SPRT gira sotto controllo di tempo,
quindi quel prezzo e' gia' dentro il verdetto: non va scontato a mano.

### Perche' a tappe
Il runtime ha ~66 GB di disco contro un dataset da 88 GB, e il mount di Drive
tiene in cache locale tutto cio' che legge. Si addestra una fetta alla volta
riprendendo dal checkpoint. `STAGE_END` ferma ogni tappa al suo confine — senza,
una tappa tira dritto fino a 40 sulla propria fetta e brucia 2,5x il tempo GPU.

⚠️ **Esegui tutte e quattro le tappe, in ordine.** Saltarne una non da' errore e
la fetta sparisce in silenzio dal training: e' successo davvero, e ha invalidato
la conclusione di un esperimento per due giorni.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va letto e ristampato da Python: subprocess.run() senza capture
    # scrive sul file descriptor del KERNEL, che Colab non mostra nella cella.
    # Con la versione precedente ogni comando risultava muto e i numeri di loss
    # del training finivano nei log del runtime invece che sotto gli occhi.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + dataset ---
from google.colab import drive
drive.mount('/content/drive')

import glob
cand = glob.glob('/content/drive/MyDrive/**/hydray_v4_2754M_shuffled.bin', recursive=True)
assert cand, 'dataset non trovato su Drive'
DATA = cand[0]
SIZE = os.path.getsize(DATA)
assert SIZE == 88129087744, f'dimensione inattesa: {SIZE} (upload incompleto?)'
os.environ['DATA'] = DATA
print('dataset ok:', DATA)

# Geometria delle fette (MiB). 1000 MiB tenuti da parte come validation set:
# non entrano mai in addestramento, quindi la validation loss e' onesta.
SLICE_MIB = 20761
SKIPS     = [0, 20761, 41522, 62283]
TEST_SKIP, TEST_MIB = 83046, 1000
NET_ID    = 'hydray-1024-full'
TOTAL_SB  = 40
TRAINER   = '/content/th4/nnue/trainer'
print('fette da', SLICE_MIB, 'MiB, test set da', TEST_MIB, 'MiB')

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato
# e il nome del checkpoint non dice nulla sul contenuto.
sh('rm -rf /content/th4')
sh('git clone --depth 1 --branch nnue-1024 https://github.com/ThomasGhione/chess_engine /content/th4')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 1024;' in src, 'NON e il branch a 1024 neuroni'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'i king bucket devono restare 4'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'start_superbatch,' in tr and 'load_from_checkpoint' in tr, 'branch privo del training a tappe'
assert 'STAGE_END' in tr, 'branch senza il limite di tappa: ogni tappa tirerebbe dritto fino a 40'
print('branch nnue-1024 con 1024 neuroni, 4 king bucket e training a tappe: ok')

In [ ]:
# --- validation set (1000 MiB dalla coda, mai addestrato) ---
if not (os.path.exists('/content/test.bin') and os.path.getsize('/content/test.bin') == TEST_MIB*1024*1024):
    sh(f'dd if="$DATA" bs=1M skip={TEST_SKIP} count={TEST_MIB} of=/content/test.bin status=progress')
print('test.bin:', os.path.getsize('/content/test.bin'), 'byte')

In [ ]:
# --- una tappa: copia la fetta, poi addestra riprendendo dal checkpoint ---
def stage(n):
    skip     = SKIPS[n-1]
    start_sb = 1 + (n-1)*10
    if n == 1:
        resume = ''
    else:
        prev = f'checkpoints/{NET_ID}-{start_sb-1}'
        assert os.path.isdir(f'{TRAINER}/{prev}'), \
            f'checkpoint mancante: {prev} — la tappa precedente non ha finito'
        # <start_sb> <checkpoint>: l'indice e' ASSOLUTO, cosi' lo scheduler del
        # learning rate resta quello del run intero e il calo cade dove deve.
        resume = f'{start_sb} {prev}'
    print(f'=== TAPPA {n}: superbatch {start_sb}-{start_sb+9}, slice skip={skip} MiB ===')
    sh(f'dd if="$DATA" bs=1M skip={skip} count={SLICE_MIB} of=/content/data.bin status=progress')
    # STAGE_END ferma la tappa al suo confine. Senza, il trainer tira dritto
    # fino a 40 sulla stessa fetta: il risultato finale resta corretto, ma
    # brucia 2,5x il tempo GPU su dati che verranno buttati.
    sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
       f'TEST_PATH=/content/test.bin STAGE_END={start_sb+9} '
       f'cargo run -r --bin trainer --features cuda -- '
       f'/content/data.bin {TOTAL_SB} {NET_ID} {resume}')
    sh('df -h /content | tail -1')
print('definita.')

In [ ]:
stage(1)   # superbatch 1-10   (~5 min copia + ~20 min con la compilazione)

In [ ]:
stage(2)   # superbatch 11-20  — deve stampare 'resuming from ... at superbatch 11'

In [ ]:
stage(3)   # superbatch 21-30

In [ ]:
stage(4)   # superbatch 31-40  — qui scende il learning rate

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# payload 6.326.288 + padding a 64 byte. La rete a 512 pesa 3.163.200:
# la taglia e' il controllo piu' rapido che l'architettura sia quella giusta.
assert 6326288 <= sz < 6326288 + 64, f'taglia {sz}: NON e la rete a 1024'
print('quantised.bin:', sz, 'byte — 1024 neuroni confermati')
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\nsalvato su Drive. Scarica quantised.bin e riporta:')
print(' - training loss finale')
print(' - VALIDATION loss finale')
print(' - i sanity eval qui sopra')

## Come leggere il risultato

Confronto finale: **testa a testa** contro la rete della 3.0.0 (stessa
architettura, 1,18B). L'unica variabile e' il dataset, quindi il risultato e' il
valore dei dati in purezza — non una sottrazione fra misure indipendenti.

- **vince nettamente** → i dati rendono ancora: conviene rimettere le macchine a
  generare, e questa rete diventa candidata per una 3.1.0
- **pareggia** → il dataset e' saturo, generare altri miliardi non paga, e cade
  anche l'ipotesi che la mappa a 8 bucket fosse solo affamata

La **validation loss**, confrontata con quella del run a 8 bucket sullo stesso
test set, dice in piu' quale architettura generalizza meglio a parita' di dati.